In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [3]:
documents = documents_llm

In [4]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [5]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [7]:
import json

user_prompt = json.dumps(doc)

In [8]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [9]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [10]:
result = response.output_parsed

print(result)

questions=['Why doesn’t my homework result match any of the multiple-choice options, and what should I check first?', 'Could the issue be caused by slicing the wrong columns or filtering after using head() or values()?', 'When should I apply a log transform in the homework, and can doing it at the wrong step change the answer?', 'Is it bad to round intermediate calculations instead of only the final result for the homework?', 'Can different sklearn, numpy, or split methods like train_test_split versus np.random.shuffle make my answer differ from the options?']


In [11]:
print(result.questions)

['Why doesn’t my homework result match any of the multiple-choice options, and what should I check first?', 'Could the issue be caused by slicing the wrong columns or filtering after using head() or values()?', 'When should I apply a log transform in the homework, and can doing it at the wrong step change the answer?', 'Is it bad to round intermediate calculations instead of only the final result for the homework?', 'Can different sklearn, numpy, or split methods like train_test_split versus np.random.shuffle make my answer differ from the options?']


In [12]:
from evaluation_utils import llm_structured

In [13]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

["Why doesn't my homework result match any of the multiple-choice answers?", 'What are the most common reasons a homework answer ends up not matching the options?', 'Could the issue be from slicing the wrong columns or applying a filter too late?', 'Can rounding too early or using a log transform in the wrong place change the answer?', 'Does a different sklearn, numpy, or split method affect the expected homework result?']


In [15]:
usage.input_tokens, usage.output_tokens

(352, 94)

In [ ]:
from evaluation_utils import calc_price
# price helper from evaluation_utils package

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.000264, 'output_cost': 0.000423, 'total_cost': 0.000687}

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': "Why doesn't my homework result match any of the multiple-choice answers?",
  'document': 'ab183bd688'},
 {'question': 'What are the most common reasons a homework answer ends up not matching the options?',
  'document': 'ab183bd688'},
 {'question': 'Could the issue be from slicing the wrong columns or applying a filter too late?',
  'document': 'ab183bd688'},
 {'question': 'Can rounding too early or using a log transform in the wrong place change the answer?',
  'document': 'ab183bd688'},
 {'question': 'Does a different sklearn, numpy, or split method affect the expected homework result?',
  'document': 'ab183bd688'}]

In [19]:
import pandas as pd

In [ ]:
pd.DataFrame(records)
# can save it to .csv later

,question,document
0,Why doesn't my homework result match any of th...,ab183bd688
1,What are the most common reasons a homework an...,ab183bd688
2,Could the issue be from slicing the wrong colu...,ab183bd688
3,Can rounding too early or using a log transfor...,ab183bd688
4,"Does a different sklearn, numpy, or split meth...",ab183bd688


In [21]:
from evaluation_utils import llm_structured_retry

In [22]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [23]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [24]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [25]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
# split them into lists
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [27]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07675724999999999

In [28]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07675724999999999

In [29]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [32]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)